you have a router llm that can decide which node to choose as the next step.

In [6]:
# import
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from typing import Literal
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from IPython.display import Image, display
import gradio as gr
import uuid
from dotenv import load_dotenv

In [7]:
load_dotenv(override=True)

True

In [ ]:
# nodes descriptions
nodes_desc = {
    'medical_unstructured_vector_database': "vector database including all the unstructured medical data, such as medical images, doctor's notes, etc.",
    'injury_description_vector_database': "vector database including detailed description of the injury, including symptoms, duration, severity, mechanism of injury",
    'medical_events_node': "a node to chronological order of medical events in claim_number when a user asks about the medical history of the claim_number"
}

# define node names as constants
ROUTER_LLM_NODE = "router_llm_node"
PRINT_NODE = "print_node"
MEDICAL_UNSTRUCTURED_VECTOR_DATABASE_NODE = "medical_unstructured_vector_database"
INJURY_DESCRIPTION_VECTOR_DATABASE_NODE = "injury_description_vector_database"
MEDICAL_EVENTS_NODE = "medical_events_node"

def format_nodes_desc(nodes_desc: Dict[str, str]) -> str:
    return "list of available nodes:\n" + "\n".join([f"{node_name}: {desc}" for node_name, desc in nodes_desc.items()])

def get_node_names_list(nodes_desc: Dict[str, str]) -> List[str]:
    return list(nodes_desc.keys())

nodes_list = get_node_names_list(nodes_desc)
# Build a Literal type from the list (must be tuple for typing)
RouterNodeLiteral = Literal[*tuple(nodes_list)]


# define schema for structured output of the router LLM
class RouterOutput(BaseModel):
    next_node: RouterNodeLiteral = Field(description="the node that the router LLM decides to route to, must be one of: " + ", ".join(nodes_list))


# define graph state: messages, router_llm_node
class State(TypedDict):
    messages: Annotated[list[Any], add_messages]
    user_query: str
    router_llm_output: RouterOutput 
    query_reformulation_bool: bool = False



# llm definition
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# structured output llm definition
structured_router_llm = llm.with_structured_output(RouterOutput)

user_query = "What is the medical history of claim_number 12345?"


def router_llm_node(state: State) -> State:
# def router_llm_node(user_query: str) -> RouterOutput:

    system_message_content = f"""
    "You are a router LLM in a LangGraph system. Your task is to route the user's query to the most appropriate node in the graph based on the content of the query.
    you are given a list of available nodes and their descriptions:
    {format_nodes_desc(nodes_desc)}

    based on the given descriptions of the nodes, decide which node is the most appropriate to handle the user's query'.
    """
    system_message = SystemMessage(content=system_message_content)

    # user_query = state['user_query']
    user_message_content = f"""
    you need to find the most appropriate node to handle the following user query:
    {user_query}
    """
    user_message = HumanMessage(content=user_message_content)
    messages = [system_message, user_message]

    # call the router LLM to get the next node decision
    router_output = structured_router_llm.invoke(messages)

    # you add the router_output to the messages of graph state to keep track of all the conversation history.
    ai_message = AIMessage(content=f"The router LLM output for this user query: {router_output.next_node}")
    
    return {
        "messages": [ai_message],
        "router_llm_output": router_output
    } # you can partially update graph state (only a number of attrs), and you only pass the ai_message in the new message item.

    # return router_output

# router_output = router_llm_node(user_query)
# print(router_output)


# define print node to print the output of condition function for testing
def print_node(state: State) -> State:
    print('print_node ...')
    print(f"Router LLM decision: (next node to route to): {state['router_llm_output'].next_node}")
    return state


# define condition function based on outer LLM decision
def router_condition(state: State) -> str:
    """
    this function is used to determine which node to route to based on the router LLM's decision stored in the graph state.
    """
    next_node = state['router_llm_output'].next_node

    if next_node == MEDICAL_UNSTRUCTURED_VECTOR_DATABASE_NODE:
        return MEDICAL_UNSTRUCTURED_VECTOR_DATABASE_NODE
    elif next_node == INJURY_DESCRIPTION_VECTOR_DATABASE_NODE:
        return INJURY_DESCRIPTION_VECTOR_DATABASE_NODE
    elif next_node == MEDICAL_EVENTS_NODE:
        return MEDICAL_EVENTS_NODE
    else:
        raise ValueError(f"Invalid router LLM output: {next_node}")
    

graph_builder  = StateGraph(State)

# add nodes
graph_builder.add_node(ROUTER_LLM_NODE, router_llm_node)
graph_builder.add_node(PRINT_NODE, print_node)

# add edges
graph_builder.add_edge(START, ROUTER_LLM_NODE)
graph_builder.add_conditional_edges(ROUTER_LLM_NODE, 
                                    router_condition, 
                                    {
    MEDICAL_UNSTRUCTURED_VECTOR_DATABASE_NODE: PRINT_NODE,
    INJURY_DESCRIPTION_VECTOR_DATABASE_NODE: PRINT_NODE,
    MEDICAL_EVENTS_NODE: PRINT_NODE
})
graph_builder.add_edge(PRINT_NODE, END)

# build graph
graph = graph_builder.compile()

# invoke the graph with the initial graph state including the user query
graph.invoke({
    "user_query": user_query
})

print_node ...
Router LLM decision (next node to route to): medical_events_node


/home/alin/miniconda3/envs/langchain/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouterOutput(next_node='medical_events_node'), input_type=RouterOutput])
  return self.__pydantic_serializer__.to_python(


{'messages': [AIMessage(content='The router LLM output for this user query: medical_events_node', additional_kwargs={}, response_metadata={}, id='14644a60-c1ce-4ba8-acee-e76388df7307', tool_calls=[], invalid_tool_calls=[])],
 'user_query': 'What is the medical history of claim_number 12345?',
 'router_llm_output': RouterOutput(next_node='medical_events_node')}

In [ ]:
# nodes definition

def medical_unstructured_vector_database(state: State) -> State:
    pass

In [ ]:
def router(state: State) -> State:
    pass